In [23]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

In [24]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password="swapnil@123",
    host="127.0.0.1",
    port=5432,
    database="bluestock_mf"
)

engine = create_engine(url)

In [25]:
connection = engine.connect()

print("Connected Successfully!")

connection.close()

Connected Successfully!


In [3]:
fund_master = pd.read_sql(
    "SELECT * FROM fund_master_features",
    engine
)

scheme_performance = pd.read_sql(
    "SELECT * FROM scheme_performance_features",
    engine
)

investor_transactions = pd.read_sql(
    "SELECT * FROM investor_transactions_features",
    engine
)

portfolio_holdings = pd.read_sql(
    "SELECT * FROM portfolio_holdings_features",
    engine
)

monthly_sip = pd.read_sql(
    "SELECT * FROM monthly_sip_features",
    engine
)

industry_folio = pd.read_sql(
    "SELECT * FROM industry_folio_features",
    engine
)

In [4]:
#Total number of funds

total_funds = fund_master['scheme_name'].nunique()
print("Total Funds:", total_funds)

Total Funds: 40


In [5]:
#Average Expense Ratio

avg_expense_ratio = round(
    fund_master['expense_ratio_pct'].mean(),2
)

print(avg_expense_ratio)

1.24


In [6]:
#Average Fund Age

avg_fund_age = round(
    fund_master['fund_age_years'].mean(),2
)

print(avg_fund_age)

19.35


In [7]:
#Average 1-Year Return

avg_return_1yr = round(
    scheme_performance['return_1yr_pct'].mean(),2
)

print(avg_return_1yr)

14.38


In [8]:
#Average 5-Year Retrun

avg_return_5yr = round(
    scheme_performance['return_5yr_pct'].mean(),2
)

print(avg_return_5yr)

14.52


In [9]:
#BEst Performing Fund

best_fund = (
    scheme_performance
    .sort_values('return_5yr_pct',
                 ascending=False)
    [['scheme_name','return_5yr_pct']]
    .head(1)
)

best_fund

,scheme_name,return_5yr_pct
29,ABSL Small Cap Fund - Regular - Growth,23.8


In [10]:
#Total Investment Amount

total_investment = (
    investor_transactions['amount_inr']
    .sum()
)

print(total_investment)

3521580430


In [11]:
#NUmber of Investors

num_investors = (
    investor_transactions['investor_id']
    .nunique()
)

print(num_investors)

5000


In [12]:
#Average transaction Size

avg_transaction = round(
    investor_transactions['amount_inr']
    .mean(),2
)

print(avg_transaction)

107437.32


In [13]:
#Total Portfolio Value

portfolio_value = round(
    portfolio_holdings['market_value_cr']
    .sum(),2
)

print(portfolio_value)

324848.62


In [14]:
#Number of stocks

total_stocks = (
    portfolio_holdings['stock_name']
    .nunique()
)

print(total_stocks)

30


In [16]:
#Top Sector


top_sector = (
    portfolio_holdings
    .groupby('sector')['market_value_cr']
    .sum()
    .sort_values(ascending=False)
    .head(1)
)

top_sector

sector
Banking    62840.29
Name: market_value_cr, dtype: float64

In [17]:
#Total SIP Inflow

total_sip = (
    monthly_sip['sip_inflow_crore']
    .sum()
)

print(total_sip)

939721


In [18]:
#Average SIP growth

avg_growth = round(
    monthly_sip['mom_growth_pct']
    .mean(),2
)

print(avg_growth)

2.16


In [19]:
#Average Equity Share%

avg_equity_share = round(
    industry_folio['equity_share_pct']
    .mean(),2
)

print(avg_equity_share)

70.0


In [20]:
kpi_df = pd.DataFrame({

'Metric':[
'Total Funds',
'Average Expense Ratio',
'Average Fund Age',
'Average 1-Year Return',
'Average 5-Year Return',
'Total Investment Amount',
'Number of Investors',
'Average Transaction Size',
'Total Portfolio Value',
'Total SIP Inflow',
'Average Equity Share %'
],

'Value':[
total_funds,
avg_expense_ratio,
avg_fund_age,
avg_return_1yr,
avg_return_5yr,
total_investment,
num_investors,
avg_transaction,
portfolio_value,
total_sip,
avg_equity_share
]

})

kpi_df

,Metric,Value
0,Total Funds,4.000000e+01
1,Average Expense Ratio,1.240000e+00
2,Average Fund Age,1.935000e+01
3,Average 1-Year Return,1.438000e+01
4,Average 5-Year Return,1.452000e+01
5,Total Investment Amount,3.521580e+09
6,Number of Investors,5.000000e+03
7,Average Transaction Size,1.074373e+05
8,Total Portfolio Value,3.248486e+05
9,Total SIP Inflow,9.397210e+05


In [21]:
kpi_df.to_sql(
    "kpi_table",
    engine,
    if_exists="replace",
    index=False
)

11